In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

import sys
from pathlib import Path

# add repo root so swiss_roll_models can be imported from anywhere
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "swiss_roll_models").exists():
        sys.path.insert(0, str(p))
        break

from swiss_roll_models.environment.dataset import generate_swiss_roll
from swiss_roll_models.environment.hilbert_distance import hilbert_distance
from swiss_roll_models.environment.traject import analyze_param_trajectory


# positive linear model = softplus(theta) ?? R^D_{>0}
class PositiveLinear(nn.Module):
    def __init__(self, D):
        super().__init__()
        self.theta = nn.Parameter(torch.zeros(D))  # trainable parameters
    def forward(self, X):
        w = F.softplus(self.theta)
        return X @ w

    def positive_params_vector(self):
        # Return the current positive parameter vector for Hilbert distance calculation
        return F.softplus(self.theta).detach().clone()


In [8]:
def run_experiment_on_subset(
    X_full, y_full, n,
    num_epochs=500,
    lr=1e-2,
    l2_reg=1e-3,
    device="cuda"
):
    N, D = X_full.shape
    idx = torch.randperm(N, device=device)[:n]
    X = X_full[idx]
    y = y_full[idx]

    dataset = TensorDataset(X, y)
    loader = DataLoader(dataset, batch_size=n, shuffle=False)  # full-batch

    model = PositiveLinear(D).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    param_traj = []
    loss_traj = []

    for epoch in range(num_epochs):
        for batch_X, batch_y in loader:
            optimizer.zero_grad()

            # 1. prediction
            y_pred = model(batch_X)

            # 2. MSE part
            mse = F.mse_loss(y_pred, batch_y)

            # 3. L2 regularization: regularize w = softplus(theta) in the positive cone
            w = F.softplus(model.theta)
            l2 = (w ** 2).sum()

            # 4. Total loss = mse + ? ||w||^2
            loss = mse + l2_reg * l2

            loss.backward()
            optimizer.step()

        loss_traj.append(loss.item())
        param_traj.append(model.positive_params_vector().cpu())

    w_star = param_traj[-1]
    w_init = param_traj[0]

    return {
        "n": n,
        "loss_traj": loss_traj,
        "param_traj": param_traj,
        "w_star": w_star,
        "w_init": w_init,
    }


In [10]:
def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Using device:", device)

    # ----- Generate Swiss roll -----
    N = 10_000
    D = 32
    X_full, y_full, u, v = generate_swiss_roll(
        n_samples=N,
        D=D,
        noise=0.0,
        device=device,
    )
    print(f"Full dataset: X={X_full.shape}, y={y_full.shape}")

    # ----- Different small sample sizes -----
    n_list = [50, 100, 200, 500]
    num_epochs = 500
    lr = 1e-2
    l2_list = [0.0, 1e-4, 1e-3, 1e-2]

    # all_results[n][l2_reg] = corresponding experiment results
    all_results = {}

    for n in n_list:
        all_results[n] = {}
        print(f"
================ n = {n} ================")

        for l2_reg in l2_list:
            print(f"
--- Running experiment (n={n}, l2_reg={l2_reg}) ---")
            res = run_experiment_on_subset(
                X_full,
                y_full,
                n=n,
                num_epochs=num_epochs,
                lr=lr,
                l2_reg=l2_reg,
                device=device,
            )

            analysis = analyze_param_trajectory(
                res["param_traj"],
                w_star=res["w_star"],
                threshold=1e-10,
            )
            res.update(analysis)
            all_results[n][l2_reg] = res

            print(f"Final loss: {res['loss_traj'][-1]:.6f}")
            print(f"Initial d_H(w_t, w*): {res['hilbert_to_final'][0]:.6f}")
            print(f"Final   d_H(w_t, w*): {res['hilbert_to_final'][-1]:.6f}")

            hilbert = res["hilbert_to_final"]
            ratio_to_prev = [hilbert[i] / hilbert[i-1] if hilbert[i-1] != 0 else float("inf") for i in range(1, len(hilbert))]
            init_dist = hilbert[0]
            ratio_to_init = [d / init_dist if init_dist != 0 else float("inf") for d in hilbert]

            print("First 15 ratio d_H(w_t, w*)/d_H(w_{t-1}, w*):", ratio_to_prev[:15])
            print("First 15 ratio d_H(w_t, w*)/d_H(w_0, w*):", ratio_to_init[:15])
            hil= res["hilbert_to_final"][:15]
            print("First 15 d_H(w_t, w*):", hil)
            print("First 15 d_H(w_t, w_0):", res["hilbert_to_init"][:15])
            print("First 15 d_H(w_{t+1}, w_t):", res["hilbert_between"][:15])
    torch.save(all_results, "swiss_roll_cone_l2_experiments.pt")


Using device: cuda
Full dataset: X=torch.Size([10000, 32]), y=torch.Size([10000])

================ n = 50 ================

--- Running experiment (n=50, l2_reg=0.0) ---
Final loss: 0.000581
Initial d_H(w_t, w*): 0.883869
Final   d_H(w_t, w*): 0.000000
First 15 ratio d_H(w_t, w*)/d_H(w_{t-1}, w*): [0.732930083797372, 0.6849496187246067, 0.702392002568377, 0.7034124898639862, 0.7063064776412671, 0.707578823500202, 0.7085784899244815, 0.7091780288692974, 0.7095828398459236, 0.7098386351768906, 0.7099967235837089, 0.7100958262862627, 0.7101513976112055, 0.7101793963329436, 0.710195225846135]
First 15 ratio d_H(w_t, w*)/d_H(w_0, w*): [1.0, 0.732930083797372, 0.502020181448804, 0.3526149605775654, 0.2480337673831566, 0.17518785657649075, 0.12395921744791544, 0.08783483511146435, 0.06229053523040804, 0.0442002948843155, 0.031375076995098616, 0.022276201868706616, 0.01581823797247881, 0.01123334380390247, 0.007977689321455868]
First 15 d_H(w_t, w*): [0.8838692903518677, 0.6478143930435181, 0

In [ ]:
if __name__ == "__main__":
    main()
